# Schedule Generation

This notebook walks through the `schedules` package — a library for generating accrual schedules used in fixed income and derivatives pricing.

An **accrual schedule** is a sequence of periods over which interest accrues. Each period has:
- an **accrual start** and **accrual end** date (the window over which interest builds up)
- a **pay date** (when the payment is made, adjusted for weekends and holidays)
- a **day count fraction (DCF)** — the fraction of the year the period represents, used to compute the actual cash flow

These schedules are the foundation of instruments like interest rate swaps (IRS), bonds, and floating rate notes.

## Setup

The notebook imports from the project root. Run Jupyter from the project root, or run the cell below which adds the parent directory to the Python path.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
root = Path.cwd().parent if Path.cwd().name == 'examples' else Path.cwd()
sys.path.insert(0, str(root))

# Use an isolated demo database — quant.db is never touched by this notebook
from database import set_db_path
set_db_path(str(root / 'examples' / 'demo.db'))

In [ ]:
from datetime import date

from schedules import (
    Schedule,
    Frequency,
    DayCountConvention,
    BusinessDayConvention,
    CalendarType,
    StubType,
)

## 1. Basic Schedule

We start with the simplest possible schedule: a 2-year semi-annual schedule with no business day adjustment.

| Parameter | Value |
|---|---|
| Effective date | 20 March 2024 |
| Termination date | 20 March 2026 |
| Frequency | Semi-annual (every 6 months) |
| Day count | ACT/360 |
| Business day convention | Unadjusted |
| Calendar | USD |

In [ ]:
sch = Schedule(
    effective_date=date(2024, 3, 20),
    termination_date=date(2026, 3, 20),
    frequency=Frequency.SEMI_ANNUAL,
    day_count_convention=DayCountConvention.ACT_360,
    business_day_convention=BusinessDayConvention.UNADJUSTED,
    calendar=CalendarType.USD,
)

periods = sch.generate()

print(f"{'Accrual Start':<16} {'Accrual End':<14} {'Pay Date':<12} {'DCF':>10}")
print("-" * 58)
for p in periods:
    print(f"{str(p.accrual_start):<16} {str(p.accrual_end):<14} {str(p.pay_date):<12} {p.dcf:>10.6f}")

print(f"\nTotal DCF: {sum(p.dcf for p in periods):.6f}")

### Understanding the Period

Each row above is a `Period` — a frozen dataclass with four fields:

- **`accrual_start`** / **`accrual_end`**: the raw (unadjusted) start and end of the accrual window
- **`pay_date`**: the business-day-adjusted payment date
- **`dcf`**: the day count fraction — **(accrual_end − accrual_start) / day count basis**

With `UNADJUSTED`, pay dates equal accrual end dates. We'll see them diverge when we apply a business day convention.

## 2. Frequency

The `Frequency` enum controls how often periods roll. Internally, it stores the number of months per period.

| Frequency | Months per period |
|---|---|
| `DAILY` | 1 business day |
| `MONTHLY` | 1 |
| `QUARTERLY` | 3 |
| `SEMI_ANNUAL` | 6 |
| `ANNUAL` | 12 |

In [ ]:
for freq in [Frequency.MONTHLY, Frequency.QUARTERLY, Frequency.SEMI_ANNUAL, Frequency.ANNUAL]:
    sch = Schedule(
        effective_date=date(2024, 1, 1),
        termination_date=date(2025, 1, 1),
        frequency=freq,
        day_count_convention=DayCountConvention.ACT_360,
        business_day_convention=BusinessDayConvention.UNADJUSTED,
        calendar=CalendarType.USD,
    )
    periods = sch.generate()
    print(f"{freq.name:<15} → {len(periods):>2} period(s), total DCF = {sum(p.dcf for p in periods):.6f}")

## 3. Day Count Conventions

The day count convention determines how the fraction of a year is calculated for each period. Different markets use different conventions.

| Convention | Formula | Common use |
|---|---|---|
| `ACT/360` | actual days / 360 | Money markets, USD IRS floating leg |
| `ACT/365 Fixed` | actual days / 365 | GBP markets |
| `30/360 Bond Basis` | 30-day months / 360 | USD fixed income |
| `ACT/ACT ISDA` | actual days / actual days in year | Government bonds |

In [ ]:
# Compare DCF for the same period under different conventions
# Q1 2024 includes Feb 29 (leap year) — a good test case

start = date(2024, 1, 1)
end   = date(2024, 4, 1)  # 91 days (Jan 31 + Feb 29 + Mar 31)

print(f"Period: {start} → {end} ({(end - start).days} actual days, leap year)\n")

for dcc in DayCountConvention:
    sch = Schedule(
        effective_date=start,
        termination_date=end,
        frequency=Frequency.ANNUAL,
        day_count_convention=dcc,
        business_day_convention=BusinessDayConvention.UNADJUSTED,
        calendar=CalendarType.USD,
    )
    dcf = sch.generate()[0].dcf
    print(f"{dcc.value:<25} DCF = {dcf:.8f}")

### Why does ACT/ACT ISDA differ?

`ACT/ACT ISDA` splits periods that span a year boundary, using the actual number of days in each year as the denominator. In 2024 (leap year, 366 days), the full Q1 falls within 2024 so the denominator is 366 — giving a slightly smaller DCF than `ACT/365 Fixed`.

## 4. Business Day Conventions

When a period end or pay date falls on a weekend or holiday, it must be adjusted. The convention controls which direction to move.

| Convention | Rule |
|---|---|
| `UNADJUSTED` | No adjustment |
| `FOLLOWING` | Move to the next business day |
| `PRECEDING` | Move to the previous business day |
| `MODIFIED_FOLLOWING` | Following, unless it crosses a month end — then preceding |

**Important:** the business day convention affects **pay dates only**, not the accrual window. Accrual start/end dates are always the unadjusted schedule dates.

To see business day adjustment in action we need a seeded holiday database. The cell below seeds `demo.db` automatically on first run — re-running the notebook is safe and will not overwrite existing data.

In [ ]:
import sqlite3
from database import get_db_path
from scripts.initialise import init_db, _seed_holidays

init_db()
with sqlite3.connect(get_db_path()) as conn:
    count = conn.execute("SELECT COUNT(*) FROM holidays").fetchone()[0]
    if count == 0:
        _seed_holidays(conn)
        print(f"Seeded demo.db with holiday data.")
    else:
        print(f"demo.db already seeded ({count} holidays). Skipping.")

In [ ]:
# Jan 1 2025 is a Wednesday holiday (New Year's Day)
# Periods ending near that date will show adjusted pay dates

sch_unadj = Schedule(
    effective_date=date(2024, 7, 1),
    termination_date=date(2025, 7, 1),
    frequency=Frequency.SEMI_ANNUAL,
    day_count_convention=DayCountConvention.ACT_360,
    business_day_convention=BusinessDayConvention.UNADJUSTED,
    calendar=CalendarType.USD,
)

sch_modfol = Schedule(
    effective_date=date(2024, 7, 1),
    termination_date=date(2025, 7, 1),
    frequency=Frequency.SEMI_ANNUAL,
    day_count_convention=DayCountConvention.ACT_360,
    business_day_convention=BusinessDayConvention.MODIFIED_FOLLOWING,
    calendar=CalendarType.USD,
)

print(f"{'Accrual End':<14} {'Unadjusted Pay':<18} {'Mod. Following Pay':<18}")
print("-" * 52)
for u, m in zip(sch_unadj.generate(), sch_modfol.generate()):
    flag = " ← adjusted" if u.pay_date != m.pay_date else ""
    print(f"{str(u.accrual_end):<14} {str(u.pay_date):<18} {str(m.pay_date):<18}{flag}")

## 5. Stub Types

When the distance between effective and termination dates is not an exact multiple of the frequency, one period will be shorter (or longer) than the rest. This irregular period is called a **stub**.

| Stub type | Where | Length |
|---|---|---|
| `SHORT_BACK` *(default)* | Last period | Shorter than regular |
| `LONG_BACK` | Last period | Merged with second-to-last |
| `SHORT_FRONT` | First period | Shorter than regular |
| `LONG_FRONT` | First period | Merged with second period |

In [ ]:
# 14-month schedule with quarterly frequency → stub period required
effective    = date(2024, 1, 1)
termination  = date(2025, 3, 1)  # not a clean multiple of 3 months from Jan 1

for stub in StubType:
    sch = Schedule(
        effective_date=effective,
        termination_date=termination,
        frequency=Frequency.QUARTERLY,
        day_count_convention=DayCountConvention.ACT_360,
        business_day_convention=BusinessDayConvention.UNADJUSTED,
        calendar=CalendarType.USD,
        stub_type=stub,
    )
    periods = sch.generate()
    print(f"\n{stub.name} ({len(periods)} periods):")
    for p in periods:
        days = (p.accrual_end - p.accrual_start).days
        flag = " ← stub" if days not in (89, 90, 91, 92) else ""
        print(f"  {p.accrual_start} → {p.accrual_end}  ({days} days){flag}")

## 6. End-of-Month Convention

When the effective date falls on the last day of a month, the `end_of_month=True` flag ensures every subsequent period also ends on the last day of its month — even in months of different lengths.

In [ ]:
# Effective Jan 31 — end of month
for eom in [False, True]:
    sch = Schedule(
        effective_date=date(2024, 1, 31),
        termination_date=date(2024, 7, 31),
        frequency=Frequency.MONTHLY,
        day_count_convention=DayCountConvention.ACT_360,
        business_day_convention=BusinessDayConvention.UNADJUSTED,
        calendar=CalendarType.USD,
        end_of_month=eom,
    )
    label = "end_of_month=True " if eom else "end_of_month=False"
    dates = " | ".join(str(p.accrual_end) for p in sch.generate())
    print(f"{label}: {dates}")

With `end_of_month=False`, the schedule naively adds one month and clips to the last day when needed (e.g. Feb 29 in a leap year). With `end_of_month=True`, every period end is the true last calendar day of its month.